In [0]:
# on charge les tables
from pyspark.sql.functions import col, current_timestamp, lit

customers = spark.table("retail_dev.silver.customers")
orders = spark.table("retail_dev.silver.orders")
order_lines = spark.table("retail_dev.silver.order_lines")
products = spark.table("retail_dev.silver.stock_items")
stock_events = spark.table("retail_dev.silver.stock_events")
inventory_health = spark.table(
    "retail_dev.gold.inventory_health"
)
stockout_risk = spark.table(
    "retail_dev.gold.stockout_risk"
)

test_results = []

def record_test(test_name, invalid_rows, description):
    status = "PASS" if invalid_rows == 0 else "FAIL"

    test_results.append(
        (test_name, status, invalid_rows, description)
    )

In [0]:
#Tester les clés et les relations
# Clés clients vides
record_test(
    "customers_non_null_key",
    customers.filter(col("customer_id").isNull()).count(),
    "Chaque client doit avoir un identifiant."
)

# Doublons clients
customer_duplicates = (
    customers.count()
    - customers.select("customer_id").distinct().count()
)

record_test(
    "customers_unique_key",
    customer_duplicates,
    "customer_id doit être unique."
)

# Commandes sans client
orders_without_customer = (
    orders.select("customer_id").distinct()
    .join(
        customers.select("customer_id").distinct(),
        on="customer_id",
        how="left_anti"
    )
    .count()
)

record_test(
    "orders_customer_integrity",
    orders_without_customer,
    "Chaque commande doit correspondre à un client."
)

# Lignes sans commande
lines_without_order = (
    order_lines.select("order_id").distinct()
    .join(
        orders.select("order_id").distinct(),
        on="order_id",
        how="left_anti"
    )
    .count()
)

record_test(
    "order_lines_order_integrity",
    lines_without_order,
    "Chaque ligne doit correspondre à une commande."
)

In [0]:
# Tester les événements
invalid_events = (
    stock_events
    .filter(
        col("event_id").isNull()
        | col("event_timestamp").isNull()
        | col("store_id").isNull()
        | (~col("store_id").between(1, 10))
        | col("event_type").isNull()
        | (~col("event_type").isin(
            "SALE",
            "RESTOCK",
            "RETURN",
            "ADJUSTMENT"
        ))
        | (col("quantity_change") == 0)
        | (
            (col("event_type") == "SALE")
            & (col("quantity_change") >= 0)
        )
        | (
            col("event_type").isin("RESTOCK", "RETURN")
            & (col("quantity_change") <= 0)
        )
    )
    .count()
)

record_test(
    "stock_events_business_rules",
    invalid_events,
    "Les événements Silver doivent respecter les règles métier."
)

events_without_product = (
    stock_events.select("stock_item_id").distinct()
    .join(
        products.select("stock_item_id").distinct(),
        on="stock_item_id",
        how="left_anti"
    )
    .count()
)

record_test(
    "stock_events_product_integrity",
    events_without_product,
    "Chaque événement doit correspondre à un produit."
)

In [0]:
#Tester les tables Gold
gold_row_difference = abs(
    inventory_health.count() - products.count()
)

record_test(
    "inventory_health_product_coverage",
    gold_row_difference,
    "Tous les produits doivent apparaître dans inventory_health."
)

invalid_risk_scores = (
    stockout_risk
    .filter(
        col("risk_score").isNull()
        | (~col("risk_score").between(0, 100))
    )
    .count()
)

record_test(
    "stockout_risk_score_range",
    invalid_risk_scores,
    "Le score de risque doit être compris entre 0 et 100."
)

invalid_reorder_quantities = (
    stockout_risk
    .filter(col("recommended_reorder_quantity") < 0)
    .count()
)

record_test(
    "reorder_quantity_positive",
    invalid_reorder_quantities,
    "La quantité recommandée ne doit jamais être négative."
)

invalid_risk_levels = (
    stockout_risk
    .filter(
        ~col("risk_level").isin(
            "LOW",
            "MEDIUM",
            "HIGH",
            "CRITICAL"
        )
    )
    .count()
)

record_test(
    "valid_risk_levels",
    invalid_risk_levels,
    "Le niveau de risque doit appartenir à la liste autorisée."
)

In [0]:
#Enregistrer et afficher les tests
test_results_df = (
    spark.createDataFrame(
        test_results,
        [
            "test_name",
            "status",
            "invalid_rows",
            "description"
        ]
    )
    .withColumn("executed_at", current_timestamp())
)

(
    test_results_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(
        "retail_dev.ops.data_quality_results"
    )
)

display(test_results_df)

In [0]:
# Faire échouer le notebook si nécessaire
failed_tests = (
    test_results_df
    .filter(col("status") == "FAIL")
    .count()
)

if failed_tests > 0:
    raise Exception(
        f"{failed_tests} test(s) de qualité ont échoué."
    )

print("Tous les tests de qualité ont réussi.")